# Apple Watch Health Analytics - Declarative Pipeline

This Spark Declarative Pipeline (SDP) implements a medallion architecture for analyzing Apple Watch health data from a simulated experimental study.

## Pipeline Architecture

**Bronze Layer** → **Silver Layer** → **Silver Fixed** → **Gold Layer**

* **Bronze**: Raw CSV ingestion using Auto Loader from Unity Catalog Volume
* **Silver**: Data quality transformations (date parsing, boolean conversion, outlier handling)
* **Silver Fixed**: Cohort assignment correction
* **Gold**: Business-ready aggregations for smoking behavior analysis

## Data Quality

The pipeline includes:
* EXPECT constraints for data validation
* Outlier detection and remediation
* Schema evolution handling
* Automatic data freshness tracking

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

@dp.table(
    name="health_irreg_rhythm_bronze",
    comment="Bronze layer: Raw health data ingested from CSV files using Auto Loader"
)
def health_irreg_rhythm_bronze():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load("/Volumes/workspace/linkprojects_apple_watch/rawdata")
        .select(
            "id",
            "date",
            "smoker",
            "rings_closed",
            "watch_type",
            "diff_exer_types",
            "max_hr",
            "irreg_rhythm",
            "cohort"
        )
    )

In [0]:
@dp.materialized_view(
    name="health_irreg_rhythm_silver",
    comment="Silver layer: Cleaned and standardized health data with quality constraints"
)
@dp.expect_or_drop("valid_date", "date_fixed IS NOT NULL")
@dp.expect_or_drop("valid_smoker", "smoker_fixed IS NOT NULL")
@dp.expect_or_drop("valid_rings", "rings_closed_fixed BETWEEN 0 AND 3")
@dp.expect_or_drop("valid_heart_rate", "max_hr_fixed BETWEEN 40 AND 220")
def health_irreg_rhythm_silver():
    df = spark.read.table("health_irreg_rhythm_bronze")
    
    # Get median heart rate for replacement
    median_hr = df.filter("max_hr != 1600").selectExpr("percentile(max_hr, 0.5) as median").first()[0]
    
    return df.select(
        "id",
        "date",
        "smoker",
        "rings_closed",
        "watch_type",
        "diff_exer_types",
        "max_hr",
        "irreg_rhythm",
        "cohort",
        # Date standardization: handle multiple formats
        when(
            col("date").rlike("^[A-Za-z]{3}-[0-9]{1,2}-[0-9]{4}$"),
            to_date(col("date"), "MMM-d-yyyy")
        ).when(
            col("date").rlike("^[0-9]{1,2}/[0-9]{1,2}/[0-9]{2}$"),
            to_date(col("date"), "M/d/yy")
        ).alias("date_fixed"),
        # Boolean conversion for smoking status
        when(col("smoker") == "yes", lit(True))
        .when(col("smoker").isin("no", "n"), lit(False))
        .alias("smoker_fixed"),
        # Fix invalid ring counts (Apple Watch only has 3 rings)
        when(col("rings_closed") == 4, lit(3))
        .otherwise(col("rings_closed"))
        .alias("rings_closed_fixed"),
        # Replace physiologically impossible heart rate with median
        when(col("max_hr") == 1600, lit(median_hr))
        .otherwise(col("max_hr"))
        .alias("max_hr_fixed")
    )

In [0]:
@dp.materialized_view(
    name="health_irreg_rhythm_silver_fixed",
    comment="Silver layer (fixed): Corrected cohort assignments for data entry errors"
)
@dp.expect_or_drop("valid_cohort", "cohort_fixed BETWEEN 1 AND 5")
def health_irreg_rhythm_silver_fixed():
    return spark.read.table("health_irreg_rhythm_silver").select(
        "id",
        "date_fixed",
        "smoker_fixed",
        "rings_closed_fixed",
        "watch_type",
        "diff_exer_types",
        "max_hr_fixed",
        "irreg_rhythm",
        # Fix cohort assignment error for participant 3
        when((col("cohort") == 3) & (col("id") == 3), lit(4))
        .otherwise(col("cohort"))
        .alias("cohort_fixed")
    )

## Gold Layer: Smoking Behavior Analysis

The gold layer aggregates participant smoking status changes across the three study timepoints:

1. **Did Not Change** - Consistent smoking status throughout
2. **Started Smoking** - Non-smoker → smoker
3. **Quit Smoking** - Smoker → non-smoker  
4. **Started Then Quit** - Non-smoker → smoker → non-smoker
5. **Quit Then Restarted** - Smoker → non-smoker → smoker

In [0]:
from pyspark.sql.window import Window

@dp.materialized_view(
    name="smoking_categorized_gold",
    comment="Gold layer: Aggregated smoking behavior categories for business analysis"
)
@dp.expect_or_drop(
    "valid_category",
    "smoking_category IN ('Did Not Change', 'Started Smoking', 'Quit Smoking', 'Started Then Quit', 'Quit Then Restarted', 'Other')"
)
def smoking_categorized_gold():
    df = spark.read.table("health_irreg_rhythm_silver_fixed")
    
    # Smoking timeline: rank timepoints by date per participant
    window_spec = Window.partitionBy("id").orderBy("date_fixed")
    count_window = Window.partitionBy("id")
    
    timeline = df.select(
        "id",
        "date_fixed",
        "smoker_fixed",
        row_number().over(window_spec).alias("timepoint"),
        count("*").over(count_window).alias("total_timepoints")
    )
    
    # Smoking summary: get first, middle, last status and count times smoked
    summary = timeline.groupBy("id", "total_timepoints").agg(
        max(when(col("timepoint") == 1, col("smoker_fixed"))).alias("first_status"),
        max(when(col("timepoint") == 2, col("smoker_fixed"))).alias("middle_status"),
        max(when(col("timepoint") == col("total_timepoints"), col("smoker_fixed"))).alias("last_status"),
        sum(when(col("smoker_fixed") == True, 1).otherwise(0)).alias("times_smoked")
    )
    
    # Categorize smoking behavior
    return summary.select(
        "id",
        when(
            (col("times_smoked") == 0) | (col("times_smoked") == col("total_timepoints")),
            lit("Did Not Change")
        ).when(
            (col("first_status") == False) & (col("last_status") == True),
            lit("Started Smoking")
        ).when(
            (col("first_status") == True) & (col("last_status") == False),
            lit("Quit Smoking")
        ).when(
            (col("first_status") == False) & (col("middle_status") == True) & (col("last_status") == False),
            lit("Started Then Quit")
        ).when(
            (col("first_status") == True) & (col("middle_status") == False) & (col("last_status") == True),
            lit("Quit Then Restarted")
        ).otherwise(lit("Other")).alias("smoking_category")
    )

In [0]:
@dp.materialized_view(
    name="smoking_summary_stats",
    comment="Gold layer: Summary statistics showing distribution of smoking behavior categories"
)
def smoking_summary_stats():
    gold = spark.read.table("smoking_categorized_gold")
    total_participants = gold.select(countDistinct("id")).first()[0]
    
    return (
        gold.groupBy("smoking_category")
        .agg(count("*").alias("participant_count"))
        .withColumn(
            "percent_of_participants",
            round((col("participant_count") / lit(total_participants)) * 100, 1)
        )
        .orderBy(desc("participant_count"))
    )